# Lab 02: Webhook Integration — Build & Test a Webhook Server

**Duration**: ~20 minutes  
**Prerequisites**: Completed `04_webhook_fundamentals.ipynb`

## Learning Objectives

By the end of this notebook, you will:
- Build a Flask webhook handler with signature verification
- Run it as a background thread in Colab
- Expose it publicly with ngrok so Stripe can reach it
- Register the endpoint with Stripe and trigger real events

---

## Architecture

```
Stripe API
    │  POST /webhook  (with Stripe-Signature header)
    ▼
ngrok tunnel  (public HTTPS URL)
    │
    ▼
Flask server on localhost:4242  (background thread in this notebook)
    │
    ▼
Event handler  (verifies signature, logs events, runs your logic)
```

## Step 1: Install Dependencies

In [1]:
!pip install stripe flask pyngrok --quiet
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.5 MB/s eta 0:00:00
Dependencies installed.


## Step 2: Configure API Key

In [2]:
import stripe

try:
    from google.colab import userdata
    stripe.api_key = userdata.get('STRIPE_SECRET_KEY')
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    stripe.api_key = getpass.getpass("Paste your Stripe test secret key (sk_test_...): ")

try:
    account = stripe.Account.retrieve()
    print(f"Connected to account: {account.id}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key.")

Loaded API key from Colab Secrets.
Connected to account: acct_1RnL4mBMxfUzotEq


## Step 3: Build the Webhook Server

Study the handler below. The key parts:
1. **Parse** the raw request body as JSON
2. **Verify** the `Stripe-Signature` header with `construct_event()`
3. **Route** by `event['type']`
4. **Return `200 OK`** to acknowledge receipt

In [6]:
import json
import threading
from flask import Flask, jsonify, request

app = Flask(__name__)

# Shared state — used to inspect received events from later cells
received_events = []
WEBHOOK_SECRET = None  # Set in Step 5 after registering the endpoint


@app.route('/webhook', methods=['POST'])
def webhook():
    payload    = request.data
    sig_header = request.headers.get('Stripe-Signature')

    # --- Signature verification ---
    if WEBHOOK_SECRET:
        try:
            event = stripe.Webhook.construct_event(
                payload, sig_header, WEBHOOK_SECRET
            )
        except stripe.error.SignatureVerificationError as e:
            print(f'Signature verification failed: {e}')
            return jsonify(success=False), 400
    else:
        # Signature verification disabled — only for local development!
        try:
            event = json.loads(payload)
        except json.JSONDecodeError as e:
            print(f'Could not parse JSON: {e}')
            return jsonify(success=False), 400

    # --- Store the event for inspection ---
    received_events.append(event)

    # --- Route by event type ---
    event_type = event['type'] if isinstance(event, dict) else event.type
    event_obj  = event['data']['object'] if isinstance(event, dict) else event.data.object

    if event_type == 'payment_intent.succeeded':
        amount = event_obj['amount'] if isinstance(event_obj, dict) else event_obj.amount
        print(f'[webhook] payment_intent.succeeded — ${amount / 100:.2f}')
        # Your logic here: fulfill order, send receipt, update database, etc.

    elif event_type == 'invoice.paid':
        inv_id = event_obj['id'] if isinstance(event_obj, dict) else event_obj.id
        print(f'[webhook] invoice.paid — {inv_id}')
        # Your logic here: provision access, send invoice email, etc.

    elif event_type == 'customer.subscription.deleted':
        sub_id = event_obj['id'] if isinstance(event_obj, dict) else event_obj.id
        print(f'[webhook] customer.subscription.deleted — {sub_id}')
        # Your logic here: revoke access, send cancellation email, etc.

    else:
        print(f'[webhook] Unhandled event type: {event_type}')

    # IMPORTANT: Always return 200 to acknowledge receipt.
    # If you return a non-2xx code, Stripe will retry the event.
    return jsonify(success=True)


# Start Flask in a daemon thread so it doesn't block the notebook
flask_thread = threading.Thread(
    target=lambda: app.run(port=4242, debug=False, use_reloader=False)
)
flask_thread.daemon = True
flask_thread.start()

print("Flask webhook server started on http://localhost:4242/webhook")

Flask webhook server started on http://localhost:4242/webhook
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 4242 is in use by another program. Either identify and stop that program, or start the server with a different port.


## Step 4: Create a Public Tunnel with ngrok

Stripe needs a public HTTPS URL to send webhooks to. ngrok creates a tunnel from a public URL to your local Flask server.

> **ngrok auth token (recommended):**  
> Free ngrok accounts support persistent tunnels. Sign up at [ngrok.com](https://ngrok.com), copy your auth token, and run the cell below — or skip it to use the unauthenticated free tier (which may have connection limits).

In [4]:
# Optional: set your ngrok auth token for a more stable tunnel
# from pyngrok import ngrok
# ngrok.set_auth_token("your-ngrok-auth-token")

from pyngrok import ngrok

public_url = ngrok.connect(4242).public_url
WEBHOOK_URL = f"{public_url}/webhook"

print(f"Public webhook URL: {WEBHOOK_URL}")

ERROR:pyngrok.process.ngrok:t=2026-04-15T14:50:53+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-15T14:50:53+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-15T14:50:53+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

## Step 5: Register the Endpoint with Stripe

Now tell Stripe to send events to your public URL. The API returns a `secret` — this is the `whsec_...` value used for signature verification.

In [5]:
endpoint = stripe.WebhookEndpoint.create(
    url=WEBHOOK_URL,
    enabled_events=[
        "payment_intent.succeeded",
        "invoice.paid",
        "customer.subscription.deleted",
    ]
)

# Enable signature verification in the running Flask server
WEBHOOK_SECRET = endpoint.secret

print(f"Webhook endpoint registered: {endpoint.id}")
print(f"Webhook URL:    {endpoint.url}")
print(f"Webhook secret: {WEBHOOK_SECRET}")

NameError: name 'WEBHOOK_URL' is not defined

**Dashboard**: [Developers → Webhooks](https://dashboard.stripe.com/test/webhooks) — you should see your new endpoint.

---

## Exercise 1: Trigger a `payment_intent.succeeded` Event

In [ ]:
import time

print("Creating a payment to trigger payment_intent.succeeded...")

pi = stripe.PaymentIntent.create(
    amount=2000,
    currency="usd",
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
)

print(f"PaymentIntent created: {pi.id} | status: {pi.status}")
print("Waiting for webhook delivery...")
time.sleep(5)

print(f"\nEvents received by webhook server: {len(received_events)}")
for e in received_events:
    t = e['type'] if isinstance(e, dict) else e.type
    print(f"  - {t}")

You should see `payment_intent.succeeded` in the list above, and a log line from the Flask handler.

---

## Exercise 2: Inspect the Received Event

In [ ]:
# Find the payment_intent.succeeded event
pi_event = next(
    (e for e in received_events
     if (e['type'] if isinstance(e, dict) else e.type) == 'payment_intent.succeeded'),
    None
)

if pi_event is None:
    print("No payment_intent.succeeded event received yet. Try running the previous cell again.")
else:
    event_id   = pi_event['id']   if isinstance(pi_event, dict) else pi_event.id
    event_type = pi_event['type'] if isinstance(pi_event, dict) else pi_event.type
    obj        = pi_event['data']['object'] if isinstance(pi_event, dict) else pi_event.data.object
    amount     = obj['amount']   if isinstance(obj, dict) else obj.amount
    status     = obj['status']   if isinstance(obj, dict) else obj.status
    obj_id     = obj['id']       if isinstance(obj, dict) else obj.id

    print(f"Event ID:               {event_id}")
    print(f"Event type:             {event_type}")
    print(f"data.object.id:         {obj_id}")
    print(f"data.object.amount:     ${amount / 100:.2f}")
    print(f"data.object.status:     {status}")

---

## Exercise 3: Add a New Event Handler

Add support for `charge.refunded` by updating the Flask handler. The pattern is the same: add an `elif` block, extract the relevant fields, run your logic.

Try it yourself — add the following block to the `webhook()` function above, then restart Flask by re-running Step 3's cell:

```python
elif event_type == 'charge.refunded':
    charge_id      = event_obj['id']              if isinstance(event_obj, dict) else event_obj.id
    amount_refunded = event_obj['amount_refunded'] if isinstance(event_obj, dict) else event_obj.amount_refunded
    print(f'[webhook] charge.refunded — {charge_id} — ${amount_refunded / 100:.2f} refunded')
    # Your logic: update order status, notify customer, etc.
```

Then trigger a refund:

In [ ]:
# Create a payment and immediately refund it
pi_to_refund = stripe.PaymentIntent.create(
    amount=3000,
    currency="usd",
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
)

# Retrieve the charge ID from the PaymentIntent
pi_expanded = stripe.PaymentIntent.retrieve(pi_to_refund.id, expand=["latest_charge"])
charge_id = pi_expanded.latest_charge.id

# Issue the refund
refund = stripe.Refund.create(charge=charge_id)
print(f"Refund created: {refund.id} | status: {refund.status}")

print("Waiting for webhook...")
time.sleep(5)

print(f"\nTotal events received: {len(received_events)}")
for e in received_events:
    t = e['type'] if isinstance(e, dict) else e.type
    print(f"  - {t}")

---

## Cleanup

In [ ]:
# Delete the webhook endpoint we registered (keeps your Dashboard tidy)
try:
    stripe.WebhookEndpoint.delete(endpoint.id)
    print(f"Deleted webhook endpoint: {endpoint.id}")
except Exception as e:
    print(f"Could not delete endpoint: {e}")

# Close the ngrok tunnel
try:
    ngrok.disconnect(public_url)
    print("ngrok tunnel closed.")
except Exception as e:
    print(f"Could not close ngrok tunnel: {e}")

---

## Reference: Deploying to Production

When you deploy your webhook handler to production:

1. **Register your production URL** in the Stripe Dashboard → Developers → Webhooks
2. **Copy the signing secret** (`whsec_...`) and store it as an environment variable — never hardcode it
3. **Choose your events carefully** — only subscribe to events you actually handle
4. **Monitor the Dashboard** for failed deliveries: Stripe retries with exponential backoff for up to 3 days

```bash
# Local development (alternative to ngrok)
stripe listen --forward-to localhost:4242/webhook
# Prints: Your webhook signing secret is whsec_xxx

# Trigger test events
stripe trigger payment_intent.succeeded
stripe trigger invoice.paid
stripe trigger customer.subscription.created
```

---

## Summary

- The Flask handler follows a clear pattern: **parse → verify → route → return 200**
- **Signature verification** uses `stripe.Webhook.construct_event()` with the `whsec_...` secret
- **ngrok** creates a public tunnel so Stripe can reach `localhost` during development
- Register your endpoint with `stripe.WebhookEndpoint.create()` to get the signing secret
- Always return `200 OK` — even for event types you don't handle

## Lab 02 Complete!

You've now built a full webhook integration:
- Understood the push model and event structure (`04_webhook_fundamentals.ipynb`)
- Built a verified webhook handler and tested it end-to-end (this notebook)

### Explore Further

- [Stripe Webhook Docs](https://docs.stripe.com/webhooks)
- [Stripe CLI reference](https://docs.stripe.com/stripe-cli)
- [Stripe Workbench](https://dashboard.stripe.com/workbench) — interactive API shell and event inspector in the Dashboard